# Transformers, TP 1 : du texte brut à un modèle bigramme

On part du texte de *Tiny Shakespeare* et on arrive à un premier modèle de langue : une table qui, pour chaque caractère, donne un score à chacun des caractères qui peuvent suivre.

Déroulé du notebook :

1. récupérer le corpus et le regarder ;
2. coder chaque caractère par un entier ;
3. fabriquer les couples (contexte, cible) ;
4. tirer des mini-batchs au hasard ;
5. écrire le modèle bigramme (avec sa méthode de génération) ;
6. générer avant l'entraînement, entraîner, puis générer à nouveau.

À rendre : le notebook exécuté de haut en bas, les deux générations (avant / après) et les réponses aux questions. Les graines aléatoires sont fixées pour que les chiffres cités dans les réponses soient reproductibles.

In [1]:
import math
from pathlib import Path
from urllib.request import urlretrieve

import torch
import torch.nn as nn
from torch.nn import functional as F

_ = torch.manual_seed(1337)

## 1. Le corpus

On télécharge le fichier une seule fois, puis on note sa longueur et on lit le début. À observer : les retours à la ligne, la ponctuation, les majuscules et les minuscules.

In [2]:
URL_CORPUS = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
fichier = Path("input.txt")
if not fichier.exists():
    urlretrieve(URL_CORPUS, fichier)

corpus = fichier.read_text(encoding="utf-8")

print(f"{len(corpus):,} caractères, {corpus.count(chr(10)):,} retours à la ligne")
print("\n----- début du texte -----\n")
print(corpus[:1000])

1,115,394 caractères, 40,000 retours à la ligne

----- début du texte -----

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for

### Questions

*Q1. Quelle est l'unité d'un token ?*
Le caractère. Une lettre, un chiffre, un signe de ponctuation, l'espace ou `\n` : chacun compte pour un token. On obtient un vocabulaire de 65 symboles (affiché à l'étape suivante).

*Q2. Pourquoi préciser `encoding="utf-8"` ?*
Sur le disque, le fichier n'est qu'une suite d'octets, et c'est l'encodage qui dit comment les transformer en caractères. Sans ce paramètre, Python prend l'encodage par défaut du système (cp1252 sous Windows par exemple) : la lecture dépend alors de la machine et peut échouer ou mal décoder certains caractères.

*Q3. Le fichier est-il déjà découpé en exemples ?*
Non, c'est un seul texte d'environ 1,1 million de caractères. Les exemples d'apprentissage n'existent pas encore : on les crée nous-mêmes en découpant des fenêtres dans ce texte (étapes 3 et 4).

## 2. Des caractères aux entiers

Chaque caractère distinct reçoit un identifiant. `encode` transforme une chaîne en liste d'entiers et `decode` fait le chemin inverse.

In [3]:
vocab = sorted(set(corpus))
vocab_size = len(vocab)

char_to_id = {c: i for i, c in enumerate(vocab)}
id_to_char = {i: c for c, i in char_to_id.items()}


def encode(texte):
    return [char_to_id[c] for c in texte]


def decode(ids):
    return "".join(id_to_char[i] for i in ids)


print(f"vocab_size = {vocab_size}")
print("vocabulaire :", repr("".join(vocab)))

message = "hii there"
ids = encode(message)
print(f"\nencode({message!r}) = {ids}")
print(f"decode(...)        = {decode(ids)!r}")
assert decode(encode(message)) == message

vocab_size = 65
vocabulaire : "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

encode('hii there') = [46, 47, 47, 1, 58, 46, 43, 56, 43]
decode(...)        = 'hii there'


### Questions

*Q1. Pourquoi trier `chars` (ici `vocab`) ?*
Un `set` n'a pas d'ordre garanti. Sans le tri, un même caractère pourrait changer d'identifiant d'une exécution à l'autre, et un modèle entraîné deviendrait inutilisable avec un nouveau dictionnaire. Le tri rend la correspondance fixe et reproductible.

*Q2. Que garantit le test `decode(encode(message)) == message` ?*
Que le codage ne perd rien : `char_to_id` et `id_to_char` sont bien inverses l'un de l'autre et chaque caractère a son propre identifiant. Le test ne vaut que pour des caractères présents dans le corpus ; un caractère inconnu déclencherait une `KeyError` dans `encode`.

## 3. Fabriquer les prédictions attendues

On coupe d'abord les données en deux : les premiers 90 % pour l'entraînement, les 10 % restants pour la validation. Ensuite, dans une fenêtre de `block_size + 1` tokens, chaque position fournit une prédiction : à partir de tout ce qui précède, deviner le token suivant. L'exemple jouet ci-dessous montre le principe : la cible `y` est l'entrée `x` décalée d'un cran.

In [4]:
suite = list(range(1, 10))
block_size = 8
print("x =", suite[:block_size])
print("y =", suite[1:block_size + 1])

x = [1, 2, 3, 4, 5, 6, 7, 8]
y = [2, 3, 4, 5, 6, 7, 8, 9]


In [5]:
data = torch.tensor(encode(corpus), dtype=torch.long)

n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"train : {len(train_data):,} tokens | validation : {len(val_data):,} tokens\n")

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    contexte, cible = x[:t + 1], y[t]
    lisible = f"{decode(contexte.tolist())!r} -> {decode([cible.item()])!r}"
    print(f"t={t}  {str(contexte.tolist()):<32} -> {cible.item():>2}    {lisible}")

train : 1,003,854 tokens | validation : 111,540 tokens

t=0  [18]                             -> 47    'F' -> 'i'
t=1  [18, 47]                         -> 56    'Fi' -> 'r'
t=2  [18, 47, 56]                     -> 57    'Fir' -> 's'
t=3  [18, 47, 56, 57]                 -> 58    'Firs' -> 't'
t=4  [18, 47, 56, 57, 58]             ->  1    'First' -> ' '
t=5  [18, 47, 56, 57, 58, 1]          -> 15    'First ' -> 'C'
t=6  [18, 47, 56, 57, 58, 1, 15]      -> 47    'First C' -> 'i'
t=7  [18, 47, 56, 57, 58, 1, 15, 47]  -> 58    'First Ci' -> 't'


### Questions

*Q1. Combien de prédictions produit ce bloc ?*
`block_size`, donc 8. Une fenêtre de 9 tokens donne 8 couples (contexte, cible), avec des contextes de 1 à 8 tokens.

*Q2. Pourquoi ne pas mélanger train et validation ?*
La validation sert à mesurer ce que le modèle fait sur du texte qu'il n'a jamais vu. Si ces données passaient aussi dans l'entraînement, une loss basse pourrait simplement venir de la mémorisation, et on ne verrait plus le surapprentissage.

*Q3. Pour t = 3, quels sont le contexte et la cible ?*
Contexte `[18, 47, 56, 57]`, soit « Firs », et cible `58`, soit « t » (ligne `t=3` de l'affichage).

## 4. Tirer un mini-batch

`get_batch` choisit `batch_size` positions de départ au hasard et empile les fenêtres correspondantes. On obtient deux tenseurs de forme `(B, T)` : les entrées et les cibles décalées d'un cran.

In [6]:
batch_size = 4
block_size = 8


def get_batch(split):
    """Renvoie (x, y), deux tenseurs (batch_size, block_size) tirés de train ou de val."""
    source = train_data if split == "train" else val_data
    departs = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in departs])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in departs])
    return x, y


xb, yb = get_batch("train")
print("forme de xb :", tuple(xb.shape), "| forme de yb :", tuple(yb.shape))
print("1re séquence, entrée :", repr(decode(xb[0].tolist())))
print("1re séquence, cible  :", repr(decode(yb[0].tolist())))

forme de xb : (4, 8) | forme de yb : (4, 8)
1re séquence, entrée : "Let's he"
1re séquence, cible  : "et's hea"


### Questions

*Q1. Pourquoi retirer `block_size` à la borne du tirage ?*
Une fenêtre qui démarre en `i` lit les indices `i` à `i + block_size - 1` pour `x`, et va jusqu'à `i + block_size` pour `y` (décalage d'un cran). Il faut donc `i + block_size ≤ len(source) - 1`. Comme `torch.randint` exclut sa borne haute, tirer dans `[0, len(source) - block_size[` donne exactement les départs valides. Sinon, les fenêtres de fin de texte seraient trop courtes et `torch.stack` échouerait.

*Q2. Que changerait B = 32 ?*
On aurait 32 séquences indépendantes par pas au lieu de 4, donc 32 × 8 = 256 prédictions par mise à jour. Le gradient est une moyenne sur plus d'exemples, donc moins bruité, mais chaque pas demande plus de calcul et de mémoire. Le contexte de chaque prédiction, lui, ne change pas.

*Q3. Que changerait T = 64 ?*
Des fenêtres de 64 caractères : plus de prédictions par séquence et des contextes jusqu'à 64 caractères. Pour le bigramme, cela ne change rien à ce qu'il peut apprendre puisqu'il ne regarde que le dernier caractère. Pour le Transformer des TP suivants, c'est la mémoire disponible qui s'allonge, avec un coût d'attention qui grandit en T².

## 5. Le modèle bigramme

La table `nn.Embedding(vocab_size, vocab_size)` contient, pour chaque caractère courant (une ligne), les logits de tous les caractères suivants possibles (les colonnes). La méthode `generate` (étape 6 de l'énoncé) est écrite directement dans la classe : à chaque tour, on prend les logits de la dernière position, on les passe dans un softmax et on tire le caractère suivant.

In [7]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)   # table C x C

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)   # (B, T) -> (B, T, C)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            dernier = logits[:, -1, :]                       # (B, C) : seule la dernière position compte
            probs = F.softmax(dernier, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

### Questions

*Q1. Pourquoi une table C × C ?*
Entrée et sortie vivent dans le même vocabulaire. Le caractère courant choisit une ligne, et cette ligne contient un score pour chacun des 65 caractères qui peuvent suivre. La case (a, b) se lit donc « score de b juste après a », d'où une table 65 × 65.

*Q2. Où retrouvera-t-on d_emb et la couche `lm_head` dans le TP 4 ?*
La table unique sera coupée en deux :
`token_embedding_table = nn.Embedding(vocab_size, n_embd)` transforme chaque caractère en un vecteur de taille d_emb (64) ;
`lm_head = nn.Linear(n_embd, vocab_size)` ramène ce vecteur vers les 65 logits.
Les blocs d'attention et de MLP viennent se placer entre ces deux couches.

## 6. Générer, entraîner, regénérer

On génère une première fois avec les poids aléatoires, on entraîne la table, puis on relance exactement la même génération. Pour avoir un point de comparaison, on affiche aussi la loss de départ à côté de ln(65), la loss d'un modèle qui répondrait uniformément.

In [8]:
model = BigramLanguageModel(vocab_size)
debut = torch.zeros((1, 1), dtype=torch.long)   # le token 0 est le retour à la ligne

xb, yb = get_batch("train")
_, loss_depart = model(xb, yb)
print(f"loss avant entraînement : {loss_depart.item():.3f}   (uniforme : ln({vocab_size}) = {math.log(vocab_size):.3f})")

print("\nGénération avec les poids aléatoires :")
print(decode(model.generate(debut, max_new_tokens=100)[0].tolist()))

loss avant entraînement : 4.678   (uniforme : ln(65) = 4.174)

Génération avec les poids aléatoires :

OMKFsPPLYAHIkwLroxdT3jCjgL'IBgq$g;GT m-fhLRkBesg;-cIBvXADqt'XHEHOMT.uKlc?CbbYjGGXhoCSmajop?oTAYoYfzF


In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(1, 1001):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step in (1, 100) or step % 250 == 0:
        print(f"étape {step:>4}  loss du mini-batch = {loss.item():.3f}")

étape    1  loss du mini-batch = 4.715
étape  100  loss du mini-batch = 4.779
étape  250  loss du mini-batch = 4.520


étape  500  loss du mini-batch = 4.101


étape  750  loss du mini-batch = 4.028
étape 1000  loss du mini-batch = 4.199


`loss.item()` ne porte que sur un mini-batch de 32 prédictions, c'est donc une valeur très bruitée. On la complète par une moyenne sur 200 batchs, en train et en validation.

In [10]:
@torch.no_grad()
def loss_moyenne(model, split, n_batchs=200):
    return sum(model(*get_batch(split))[1].item() for _ in range(n_batchs)) / n_batchs


print(f"loss moyenne train : {loss_moyenne(model, 'train'):.3f}")
print(f"loss moyenne val   : {loss_moyenne(model, 'val'):.3f}")

loss moyenne train : 4.011
loss moyenne val   : 4.017


In [11]:
print("Génération après 1 000 étapes :")
print(decode(model.generate(debut, max_new_tokens=100)[0].tolist()))

Génération après 1 000 étapes :

k,VllXfK-VsqUO?LZH!P h3ZEz,xxAQS:HE-Xh zRT ys b-UCVhe
OMyytcPV b.YvWGWVKIZ
terAM3AgrdiZJMHVLENuojGCZ


### Bilan du TP 1

*Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération ?*
`logits` contient une prédiction pour chaque position, mais toutes les positions sauf la dernière prédisent des caractères qu'on connaît déjà. Seule la dernière prédit le caractère à inventer. `logits[:, -1, :]` passe de `(B, T, C)` à `(B, C)`, la forme qu'attendent `softmax` puis `multinomial`.

*Pourquoi aplatir B × T pour la loss ?*
`F.cross_entropy` attend des logits `(N, C)` et des cibles `(N,)`. Chaque position de chaque séquence est une prédiction à part entière ; on les aligne donc toutes (N = B·T) et on calcule la moyenne d'un seul coup.

*Avant / après entraînement.*
Une prédiction uniforme donnerait ln(65) ≈ 4,17. Au départ, la loss est au-dessus (4,68 ici) : la table est initialisée avec des valeurs N(0, 1), le modèle n'est pas uniforme mais « sûr de lui au hasard ». Le texte généré est alors une suite de caractères sans structure.

Après 100 étapes, la loss n'a presque pas bougé (4,78 sur le mini-batch de l'étape 100, contre 4,72 à l'étape 1). Après 1 000 étapes (`lr = 1e-3`, `batch_size = 4`), la loss moyenne tombe à 4,01 en train et 4,02 en validation : on passe sous ln(65), donc le modèle fait mieux qu'un tirage uniforme, mais la baisse reste faible. Chaque étape ne voit que 4 × 8 = 32 prédictions, et 1 000 petites mises à jour ne suffisent pas à remplir correctement 65 × 65 = 4 225 scores. Le texte reste presque aléatoire, avec un peu plus d'espaces et de retours à la ligne. Pour voir apparaître des paires de lettres plausibles, il faudrait environ 10 000 étapes ou un batch plus grand. Les valeurs train et val sont proches : pas de surapprentissage, ce qui est logique pour un modèle aussi petit et aussi peu entraîné.